# Comorbidity sub-maps around the interface proteins

One sub-map per interface protein, per pairing: what the protein is downstream of in COVID and what it drives in the comorbid disease, joined by a synthetic central node standing for the protein itself.

Two pairings, held in one `PAIRINGS` list exactly as `4_20` does for the gene-set analyses. **COVID → PD**, whose downstream side is the PD activity-flow maps, and **COVID → AD**, whose downstream side is the AD knowledge graph. Both are ordinary CellDesigner activity-flow collections — `2_10` exports the AD KG's influence-graph projection to a CellDesigner file and imports it as `AD_KG_CD_AF` — so nothing below the parameter cell distinguishes them, and neither does the library.

Nothing here writes to the database.

In [1]:
%store -r

In [2]:
import commute_dm.core
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
def make_backend():
    return momapy_kb.lpg.backends.neo4j.Neo4jBackend(
        hostname=credentials.NEO4J_URI,
        username=credentials.NEO4J_USERNAME,
        password=credentials.NEO4J_PASSWORD,
        notifications_min_severity="off",
    )

## Parameters

One entry per pairing: the collection names, the interface to join over, the levels to walk and where to write — the same shape as `4_20`'s `PAIRINGS`, so the two notebooks stay comparable.

`max_levels` differs between the two, and that is a property of the downstream graph rather than of the code. The AD influence graph is far denser than an activity-flow map: from the interface seeds the median downstream selection is 3 nodes at one hop, 6 at two and 18 at three, but 140 at four and about 960 unbounded — past three hops the maps stop being readable.

In [4]:
MIN_N_NODES = 5

PAIRINGS = [
    {
        "name": "COVID -> PD",
        # The interface is the three-way one: a protein has to be shared with
        # the AD KG as well to be drawn here.
        "upstream_collection_name": "COVID_DM_CD_AF",
        "downstream_collection_name": "PD_DM_CD_AF",
        "interface_collection_names": (
            "COVID_DM_CD_AF",
            "PD_DM_CD_AF",
            "AD_KG_CD_AF",
        ),
        "max_levels": [2, 3, 4, 5, 6],
        "graphs_dir_path": INTERFACE_PD_ANALYSIS_GRAPHS_DIR,
    },
    {
        # Two-way: the pairing *is* COVID and the AD KG, which `2_10` stores as
        # an ordinary CellDesigner collection.
        "name": "COVID -> AD",
        "upstream_collection_name": "COVID_DM_CD_AF",
        "downstream_collection_name": "AD_KG_CD_AF",
        "interface_collection_names": (
            "COVID_DM_CD_AF",
            "AD_KG_CD_AF",
        ),
        "max_levels": [1, 2, 3],
        "graphs_dir_path": INTERFACE_AD_ANALYSIS_GRAPHS_DIR,
    },
]

In [5]:
with momapy_kb.lpg.session.Session(make_backend()) as session:
    for pairing in PAIRINGS:
        pairing["interface"] = commute_dm.core.get_interface(
            session, pairing["interface_collection_names"]
        )
        (
            pairing["influences"],
            pairing["source_map"],
            pairing["node_id_to_object"],
        ) = commute_dm.core.load_submap_inputs(
            session,
            pairing["upstream_collection_name"],
            pairing["downstream_collection_name"],
        )

{
    pairing["name"]: {
        "n_interface": len(pairing["interface"]),
        "n_species": len(pairing["source_map"].model.species),
        "n_modulations": len(pairing["source_map"].model.modulations),
    }
    for pairing in PAIRINGS
}

{'COVID -> PD': {'n_interface': 127,
  'n_species': 6342,
  'n_modulations': 9886,
  'n_seeds_expanded': 0},
 'COVID -> AD': {'n_interface': 189,
  'n_species': 6148,
  'n_modulations': 8417,
  'n_seeds_expanded': 79}}

## What a BEL downstream side became

Empty for COVID → PD, which has none. See `commute_dm.bel2cd` for the four identity invariants these numbers are the evidence for.

## Assembling and writing the sub-maps

We assemble and write the sub-map upstream of the upstream seeds and downstream of the downstream seeds around each interface protein, joined by a synthetic central node.

A species is filled by **which walk reached it** — one colour for the upstream selection, another for the downstream one, a third for the central node. Every element of a CellDesigner side is a stored activity-flow element reused as it is; a BEL side contributes real activity-flow content built from its terms (badges, nested subunits, active borders, `loc()` boxes), not text boxes.

In [7]:
with momapy_kb.lpg.session.Session(make_backend()) as session:
    for pairing in PAIRINGS:
        commute_dm.utils.remake_dir(pairing["graphs_dir_path"])
        print(f"{pairing['name']} -> {pairing['graphs_dir_path']}")
        pairing["stats_df"] = commute_dm.core.make_and_write_submaps_from_interface(
            session=session,
            interface=pairing["interface"],
            influences=pairing["influences"],
            source_map=pairing["source_map"],
            node_id_to_object=pairing["node_id_to_object"],
            output_dir_path=pairing["graphs_dir_path"],
            upstream_collection_name=pairing["upstream_collection_name"],
            downstream_collection_name=pairing["downstream_collection_name"],
            max_levels=pairing["max_levels"],
            min_n_nodes=MIN_N_NODES,
        )

{pairing["name"]: len(pairing["stats_df"]) for pairing in PAIRINGS}

COVID -> PD -> ../../build/results/interface/analysis/graphs


COVID -> AD -> ../../build/results/interface/analysis_ad/graphs


{'COVID -> PD': 206, 'COVID -> AD': 101}

How many maps came out per level. `MIN_N_NODES` applies to **both** directions, and for COVID → AD it is the COVID upstream side that usually falls short at one hop — not the AD side.

In [8]:
{
    pairing["name"]: (
        pairing["stats_df"].groupby("max_level")["identifier"].count().to_dict()
    )
    for pairing in PAIRINGS
}

{'COVID -> PD': {2: 34, 3: 40, 4: 44, 5: 44, 6: 44},
 'COVID -> AD': {1: 16, 2: 40, 3: 45}}

## Read-back

Every written file is read back. This is the assertion the identity invariants exist for: a violation writes a valid-looking file and fails here with a `KeyError`, so a regression must be loud rather than discovered in CellDesigner.

In [9]:
import glob
import os.path

import momapy.io.core

read_back_failures = []
n_written_files = {}
for pairing in PAIRINGS:
    written_file_paths = sorted(
        glob.glob(os.path.join(pairing["graphs_dir_path"], "*", "*.xml"))
    )
    n_written_files[pairing["name"]] = len(written_file_paths)
    for written_file_path in written_file_paths:
        try:
            read_back = momapy.io.core.read(
                written_file_path, reader="celldesigner"
            ).obj
            assert read_back.model.species
        except Exception as exception:
            read_back_failures.append((written_file_path, repr(exception)))
assert not read_back_failures, read_back_failures[:3]
n_written_files

{'COVID -> PD': 206, 'COVID -> AD': 101}